In [1]:
!pip install -q ultralytics kaggle opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 7.4 MB/s eta 0:00:00


In [2]:
import os
from google.colab import files

# Upload your kaggle.json file when prompted
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download the PlantVillage dataset
!kaggle datasets download -d emmarex/plantdisease -p ./raw_data --unzip

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:06<00:00, 110MB/s]



In [4]:
# Cell 2: Dataset Curation and Train/Val Split
import os
import shutil
import random

# Mapping raw dataset folder names to standardized AgriVox keys
CLASS_MAPPING = {
    "Pepper__bell___Bacterial_spot": "Pepper_bell_Bacterial_spot",
    "Pepper__bell___healthy": "Pepper_bell_healthy",
    "Potato___Early_blight": "Potato_Early_blight",
    "Potato___Late_blight": "Potato_Late_blight",
    "Potato___healthy": "Potato_healthy",
    "Tomato_Early_blight": "Tomato_Early_blight",
    "Tomato_Late_blight": "Tomato_Late_blight",
    "Tomato_Leaf_Mold": "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato_Septoria_leaf_spot",
    "Tomato__healthy": "Tomato_healthy"
}

SOURCE_ROOT = "./raw_data/PlantVillage"
TARGET_ROOT = "./dataset"

# Clean target directory if re-running
if os.path.exists(TARGET_ROOT):
    shutil.rmtree(TARGET_ROOT)

# Create train and val directories for each class
for split in ['train', 'val']:
    for clean_name in CLASS_MAPPING.values():
        os.makedirs(os.path.join(TARGET_ROOT, split, clean_name), exist_ok=True)

random.seed(42)

for raw_folder, clean_folder in CLASS_MAPPING.items():
    src_path = os.path.join(SOURCE_ROOT, raw_folder)
    if not os.path.exists(src_path):
        print(f"Warning: {src_path} not found!")
        continue

    all_images = [f for f in os.listdir(src_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(all_images)

    # Cap at 800 images per class for balanced, fast on-device model training
    selected = all_images[:800]
    split_idx = int(0.8 * len(selected))

    train_imgs = selected[:split_idx]
    val_imgs = selected[split_idx:]

    for img in train_imgs:
        shutil.copy(os.path.join(src_path, img), os.path.join(TARGET_ROOT, 'train', clean_folder, img))
    for img in val_imgs:
        shutil.copy(os.path.join(src_path, img), os.path.join(TARGET_ROOT, 'val', clean_folder, img))

print("Data curation successful!\n")
print("--- DATASET VERIFICATION ---")
for split in ['train', 'val']:
    total_imgs = sum([len(files) for r, d, files in os.walk(os.path.join(TARGET_ROOT, split))])
    print(f"Total {split.upper()} images: {total_imgs}")

Data curation successful!

--- DATASET VERIFICATION ---
Total TRAIN images: 5241
Total VAL images: 1311
